# 📊 Trực Quan Hiện Trạng Lớp Silver (MinIO)

Chạy Notebook này để xem dữ liệu của từng bảng dưới dạng Dataframe trực quan (HTML Pandas).


In [1]:
import sys, os

# === Cấu hình kết nối MinIO Server ===
os.environ['MINIO_ENDPOINT'] = 'http://20.41.113.183:9000'
os.environ['MINIO_ACCESS_KEY'] = 'minioadmin'
os.environ['MINIO_SECRET_KEY'] = 'minioadmin123'

if 'SEED_PATH' in os.environ:
    del os.environ['SEED_PATH']

if 'google.colab' in sys.modules:
    !rm -rf /content/Lab
    !git clone -b ml https://github.com/nbngoc123/Lab.git /content/Lab
    os.chdir('/content/Lab/football-lake')
    sys.path.insert(0, '/content/Lab/football-lake')
    !pip install minio duckdb pandas understatapi boto3 python-dotenv -q
    print('Setup xong cho Colab!')
else:
    # Chạy trên máy ảo (VM) hoặc local
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
    print(f'Thêm {PROJECT_ROOT} vào sys.path để nhận diện thư viện lake')



Cloning into '/content/Lab'...
remote: Enumerating objects: 1021, done.
remote: Counting objects: 100% (155/155), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 1021 (delta 66), reused 110 (delta 35), pack-reused 866 (from 1)
Receiving objects: 100% (1021/1021), 75.43 MiB | 25.11 MiB/s, done.
Resolving deltas: 100% (488/488), done.
Setup xong cho Colab!


In [2]:
import os
import io
import pandas as pd
from IPython.display import display, HTML
from lake import minio_io as mio

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 10)
pd.set_option('display.width', 1000)

print('Đã load thư viện thành công. Sẵn sàng kết nối MinIO...')

Đã load thư viện thành công. Sẵn sàng kết nối MinIO...


In [3]:
files = list(mio.list_keys("silver/"))

datasets = {}
for key, size in files:
    if not key.endswith(".parquet"): continue
    parts = key.split("/")
    if len(parts) >= 3:
        ds_name = f"{parts[1]}/{parts[2]}"
        datasets.setdefault(ds_name, []).append((key, size))

for ds_name, file_list in sorted(datasets.items()):
    display(HTML(f"<h3>⚽ Bảng: <span style='color: #2c3e50;'>{ds_name}</span></h3>"))
    
    sample_key = file_list[0][0]
    try:
        raw = mio.read_bytes(sample_key)
        df = pd.read_parquet(io.BytesIO(raw))
        
        info_html = f"""
        <ul>
            <li><b>Số lượng file (partition):</b> {len(file_list)}</li>
            <li><b>Số cột:</b> {len(df.columns)}</li>
            <li><b>Số ô Null (mẫu):</b> {df.isna().sum().sum()}</li>
        </ul>
        """
        display(HTML(info_html))
        display(df.head(5))
        display(HTML("<hr>"))
        
    except Exception as e:
        display(HTML(f"<p style='color: red;'>Lỗi khi đọc: {e}</p>"))


,match_id,home_team,away_team,commence_time,bookmaker,last_update,odds_home,odds_draw,odds_away,ingest_date
0,b870dcfba5e5920d3f1551a3aa1ea435,Arsenal,Leeds United,2026-10-10 11:30:00+00:00,Unibet (UK),2026-09-22 07:58:56+00:00,1.37,4.75,7.5,2026-09-22
1,b870dcfba5e5920d3f1551a3aa1ea435,Arsenal,Leeds United,2026-10-10 11:30:00+00:00,Codere (IT),2026-09-22 07:58:55+00:00,1.38,5.00,7.8,2026-09-22
2,b870dcfba5e5920d3f1551a3aa1ea435,Arsenal,Leeds United,2026-10-10 11:30:00+00:00,Betfred (UK),2026-09-22 07:58:55+00:00,1.33,5.00,9.0,2026-09-22
3,b870dcfba5e5920d3f1551a3aa1ea435,Arsenal,Leeds United,2026-10-10 11:30:00+00:00,Unibet (FR),2026-09-22 07:58:54+00:00,1.31,4.50,7.0,2026-09-22
4,b870dcfba5e5920d3f1551a3aa1ea435,Arsenal,Leeds United,2026-10-10 11:30:00+00:00,Winamax (DE),2026-09-22 07:58:55+00:00,1.35,5.00,7.0,2026-09-22


,match_id,home_team,away_team,commence_time,bookmaker,last_update,team,point,price,ingest_date
0,b870dcfba5e5920d3f1551a3aa1ea435,Arsenal,Leeds United,2026-10-10 11:30:00+00:00,1xBet,2026-09-22 07:59:16+00:00,Arsenal,-1.50,2.15,2026-09-22
1,b870dcfba5e5920d3f1551a3aa1ea435,Arsenal,Leeds United,2026-10-10 11:30:00+00:00,1xBet,2026-09-22 07:59:16+00:00,Leeds United,1.50,1.81,2026-09-22
2,b870dcfba5e5920d3f1551a3aa1ea435,Arsenal,Leeds United,2026-10-10 11:30:00+00:00,MyBookie.ag,2026-09-22 07:58:18+00:00,Arsenal,-1.50,2.02,2026-09-22
3,b870dcfba5e5920d3f1551a3aa1ea435,Arsenal,Leeds United,2026-10-10 11:30:00+00:00,MyBookie.ag,2026-09-22 07:58:18+00:00,Leeds United,1.50,1.71,2026-09-22
4,b870dcfba5e5920d3f1551a3aa1ea435,Arsenal,Leeds United,2026-10-10 11:30:00+00:00,Pinnacle,2026-09-22 07:59:54+00:00,Arsenal,-1.25,1.85,2026-09-22


,match_id,home_team,away_team,commence_time,bookmaker,last_update,name,point,price,ingest_date
0,b870dcfba5e5920d3f1551a3aa1ea435,Arsenal,Leeds United,2026-10-10 11:30:00+00:00,Codere (IT),2026-09-22 07:58:55+00:00,Over,2.5,1.67,2026-09-22
1,b870dcfba5e5920d3f1551a3aa1ea435,Arsenal,Leeds United,2026-10-10 11:30:00+00:00,Codere (IT),2026-09-22 07:58:55+00:00,Under,2.5,2.03,2026-09-22
2,b870dcfba5e5920d3f1551a3aa1ea435,Arsenal,Leeds United,2026-10-10 11:30:00+00:00,1xBet,2026-09-22 07:59:16+00:00,Over,2.5,1.83,2026-09-22
3,b870dcfba5e5920d3f1551a3aa1ea435,Arsenal,Leeds United,2026-10-10 11:30:00+00:00,1xBet,2026-09-22 07:59:16+00:00,Under,2.5,2.14,2026-09-22
4,b870dcfba5e5920d3f1551a3aa1ea435,Arsenal,Leeds United,2026-10-10 11:30:00+00:00,MyBookie.ag,2026-09-22 07:58:18+00:00,Over,2.5,1.69,2026-09-22


,competition_id,code,name,area,type,current_season_id,current_season_start,current_matchday
0,2193,ACQ,Africa Cup Of Nations - Qualification,Africa,CUP,2545.0,2026-09-23,1.0
1,2006,QCAF,WC Qualification CAF,Africa,CUP,1609.0,2023-11-15,10.0
2,2176,AC,Africa Cup Of Nations,Africa,CUP,2385.0,2025-12-21,3.0
3,2166,ACL,AFC Champions League,Africa,CUP,1559.0,2022-04-06,NaN
4,2023,PBN,Primera B Nacional,Argentina,LEAGUE,2500.0,2026-06-17,30.0


,player_id,name,position,date_of_birth,nationality,team_id,team_name
0,7015,Matthias Köbbing,Goalkeeper,1997-05-28,Germany,1,1. FC Köln
1,9388,Ron-Robert Zieler,Goalkeeper,1989-02-12,Germany,1,1. FC Köln
2,9957,Marvin Schwäbe,Goalkeeper,1995-04-25,Germany,1,1. FC Köln
3,289578,Martin James,Goalkeeper,2008-02-14,France,1,1. FC Köln
4,309,Timo Hübers,Defence,1996-07-20,Germany,1,1. FC Köln


...

,entity_type,article,label,lang,date,views,year,month,week_of_year,day_of_week,is_weekend,football_season,views_7d_avg,views_28d_avg,views_28d_std,zscore,spike_ratio_7d,is_spike
0,team,Newcastle_United_F.C.,Newcastle United,de,2025-01-11,1,2025,1,2,5,1,2024/2025,0.142857,0.035714,0.188982,5.10252,NaN,1
1,team,Tottenham_Hotspur_F.C.,Tottenham Hotspur,de,2023-05-25,1,2023,5,21,3,0,2022/2023,0.142857,0.035714,0.188982,5.10252,NaN,1
2,team,Manchester_City_F.C.,Manchester City,de,2024-12-27,1,2024,12,52,4,0,2024/2025,0.142857,0.035714,0.188982,5.10252,NaN,1
3,team,Tottenham_Hotspur_F.C.,Tottenham Hotspur,de,2024-10-10,1,2024,10,41,3,0,2024/2025,0.142857,0.035714,0.188982,5.10252,NaN,1
4,team,Crystal_Palace_F.C.,Crystal Palace,fr,2024-12-14,1,2024,12,50,5,1,2024/2025,0.142857,0.035714,0.188982,5.10252,NaN,1


,entity_type,article,label,date,views,year,month,week_of_year,day_of_week,is_weekend,football_season,views_7d_avg,views_28d_avg,views_28d_std,zscore,spike_ratio_7d,is_spike
0,player,Alexander_Isak,Alexander Isak,2023-01-01,1628,2023,1,52,6,1,2022/2023,1628.0,1628.0,NaN,NaN,NaN,0
1,player,Alexander_Isak,Alexander Isak,2023-01-02,2048,2023,1,1,0,0,2022/2023,1838.0,1838.0,NaN,NaN,NaN,0
2,player,Alexander_Isak,Alexander Isak,2023-01-03,3986,2023,1,1,1,0,2022/2023,2554.0,2554.0,NaN,NaN,NaN,0
3,player,Alexander_Isak,Alexander Isak,2023-01-04,2372,2023,1,1,2,0,2022/2023,2508.5,2508.5,NaN,NaN,0.928739,0
4,player,Alexander_Isak,Alexander Isak,2023-01-05,1553,2023,1,1,3,0,2022/2023,2317.4,2317.4,989.869082,-0.772223,0.619095,0


,entity_type,article,label,lang,date,views,year,month,week_of_year,day_of_week,is_weekend,football_season,views_7d_avg,views_28d_avg,views_28d_std,zscore,spike_ratio_7d,is_spike
0,player,Alexander_Isak,Alexander Isak,de,2023-01-01,65,2023,1,52,6,1,2022/2023,65.00,65.00,NaN,NaN,NaN,0
1,player,Alexander_Isak,Alexander Isak,de,2023-01-02,56,2023,1,1,0,0,2022/2023,60.50,60.50,NaN,NaN,NaN,0
2,player,Alexander_Isak,Alexander Isak,de,2023-01-03,152,2023,1,1,1,0,2022/2023,91.00,91.00,NaN,NaN,NaN,0
3,player,Alexander_Isak,Alexander Isak,de,2023-01-04,86,2023,1,1,2,0,2022/2023,89.75,89.75,NaN,NaN,0.945055,0
4,player,Alexander_Isak,Alexander Isak,de,2023-01-05,51,2023,1,1,3,0,2022/2023,82.00,82.00,41.358191,-0.749549,0.568245,0


,comment_id,video_id,author,text,like_count,published_at,published_ts,ingest_date
0,UgwhJaZNIMg6dBgnLt54AaABAg,GAvCftCXL0E,@brunosantos6595,"Congratulations Liverpool, great victory! Gree...",1,2026-09-21T08:30:46Z,2026-09-21 08:30:46+00:00,2026-09-21
1,Ugy5-dq4aFWDKyKyeeB4AaABAg,GAvCftCXL0E,@ooolega,Witz is threre! as always my man,0,2026-09-21T08:23:53Z,2026-09-21 08:23:53+00:00,2026-09-21
2,UgyAL_rAcCrVBma-ZFB4AaABAg,GAvCftCXL0E,@UnbelievableJeff,i love Liverpool but i'd rather watch Brighton...,0,2026-09-21T08:19:09Z,2026-09-21 08:19:09+00:00,2026-09-21
3,UgwyB3hcYEBK-YaT2Ul4AaABAg,GAvCftCXL0E,@litsitsomf2,Allison becker has been sooooo good this seaso...,0,2026-09-21T08:18:58Z,2026-09-21 08:18:58+00:00,2026-09-21
4,UgwfIwxrpYKOeVSbt7B4AaABAg,GAvCftCXL0E,@johnyboy2934,"i mean 2026 is full of surprises , who thought...",0,2026-09-21T08:16:59Z,2026-09-21 08:16:59+00:00,2026-09-21


,video_id,query,published_at,channel_id,title,description,channel_title,published_ts,ingest_date
0,GAvCftCXL0E,Ngoại hạng Anh highlight,2026-09-20T21:00:28Z,UC9LQwHZoucFT94I2h6JOcjw,Highlights: Bournemouth 0-1 Liverpool | Isak G...,Watch highlights from Liverpool's 1-0 victory ...,Liverpool FC,2026-09-20 21:00:28+00:00,2026-09-21
1,EFPDyDeB2HI,Ngoại hạng Anh highlight,2026-09-20T12:59:55Z,UCbkllEQivzBHCkaQX-w4N3Q,Highlights: Thành Phố Đồng Nai - Công An Hà Nộ...,"Xem trọn Ngoại Hạng Anh, Truyền Hình, Giải Trí...",BÓNG ĐÁ VIỆT NAM,2026-09-20 12:59:55+00:00,2026-09-21
2,gIPTUHoGnDM,Ngoại hạng Anh highlight,2026-09-20T14:56:34Z,UCrVFnGI-iQLDHpwxUe1Q8Jg,Manchester City vs Sunderland Highlights 5-3 C...,Manchester City vs Sunderland Highlights | CHE...,Footballia,2026-09-20 14:56:34+00:00,2026-09-21
3,ctznPsjTgi8,Ngoại hạng Anh highlight,2026-09-20T12:22:41Z,UCbkllEQivzBHCkaQX-w4N3Q,Highlights: Hoàng Anh Gia Lai - Hưng Yên | Hiệ...,"Xem trọn Ngoại Hạng Anh, Truyền Hình, Giải Trí...",BÓNG ĐÁ VIỆT NAM,2026-09-20 12:22:41+00:00,2026-09-21
4,jbXc4VgoglA,Ngoại hạng Anh highlight,2026-09-13T21:00:37Z,UCG5qGWdu8nIRZqJ_GgDwQ-w,EVERY Weekend Goal | Matchweek 4 | 2026/27 Pre...,Love this video? We've got even more for you t...,Premier League,2026-09-13 21:00:37+00:00,2026-09-21
